# TissueMC: GPU rung R8 on a Colab T4

R7 keeps the depth and radius profiles in a private copy per warp in shared memory. Its counters showed what is left: 633 million shared-memory compare-and-swap iterations, about 19 per intended add, because lanes of a warp that add to the same few bins retry against each other; 5.1 billion extra warp instructions; and the double-precision pipe 77% busy.

**R8 combines the lanes of a warp that share a bin before adding**: they sum their weights (with warp shuffles) and one lane per bin makes the single add, so there is nothing to retry against inside the warp. What is computed does not change; only how many adds are made.

This notebook validates R8, compares it with the CPU on identical photons (including the depth and radius profiles; R7 is compared too), benchmarks **R7, R8, the profiles-off diagnostic (the target, not a result) and R8 again in the same session**, and collects every Nsight Compute counter for R7 and R8 so the retry loops can be counted. It ends with a summary block and a digest for you to paste back.

**Before you start:** *Runtime → Change runtime type → T4 GPU.* Nothing else to record.

**Step 1 is a short check.** Every later step starts only if it passed; otherwise it prints "SKIPPED". If step 1 fails, **paste its output**. Raw counters are kept in `results/*.ncu.csv` and reused if you run a step again (a session that still has R7's from last time will reuse them). Expect about 25 minutes. Nothing here needs passwords or keys.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
# Safe to run twice: reuse the clone if it is already there instead of nesting a second one.
!if [ -d /content/tissuemc/.git ]; then git -C /content/tissuemc pull -q; else git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git /content/tissuemc; fi
%cd /content/tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
# Build. The output goes to files so that the summary at the end can show it if something fails.
!mkdir -p results
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > results/configure.log 2>&1; tail -3 results/configure.log
!cmake --build build > results/build.log 2>&1 || echo "BUILD FAILED" >> results/build.log; tail -12 results/build.log
!./build/tissuemc --version

In [ ]:
# 1. Quick check: does R8, with its depth and radius profiles, agree with the CPU on identical
# photons? The later steps run only if this passes (it leaves results/.quick_ok). The output is
# kept in results/quick.log so that the summary can show why if it fails.
!rm -f results/.quick_ok results/quick.log
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 8 --photons 200000 > results/quick.log 2>&1 && touch results/.quick_ok; tail -30 results/quick.log

In [ ]:
# 2. Validate rung 8 against the published values and the independent oracle.
!if [ -f results/.quick_ok ]; then tissuemc-validate --binary build/tissuemc --engine cuda --rung 8 --report results/gpu-r8-validation.json 2>&1 | grep -v '^  note' | tail -30; else echo "SKIPPED: the quick check did not pass, so validation of rung 8 was not run"; fi

In [ ]:
# 3. Same photons through the CPU (float32) and rungs 7 and 8: R_d, T_t, absorbed weight and the
# depth and radius profiles.
!if [ -f results/.quick_ok ]; then tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 7 --photons 2000000 --report results/gpu-r7-r8session-cross-engine.json; else echo "SKIPPED: the quick check did not pass, so the rung 7 comparison was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 8 --photons 2000000 --report results/gpu-r8-cross-engine.json; else echo "SKIPPED: the quick check did not pass, so the rung 8 comparison was not run"; fi

In [ ]:
# 4. Benchmarks back to back: R7, R8, the profiles-off diagnostic (the target), then R8 again.
# About 8 minutes.
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 7 --binary build/tissuemc --output results/gpu-r7-r8session.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 7 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 8 --binary build/tissuemc --output results/gpu-r8.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 8 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 6 --no-profiles --binary build/tissuemc --output results/gpu-r6-noprofiles-r8session.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the diagnostic benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 8 --binary build/tissuemc --output results/gpu-r8-again.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the second rung 8 benchmark was not run"; fi

In [ ]:
# 5. Every Nsight Compute counter for R7 (reused if this session already has it) and for R8,
# 500,000 photons. About 3 to 6 minutes each.
!if [ -f results/.quick_ok ]; then tissuemc-profile --rung 7 --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r7-full.json; else echo "SKIPPED: the quick check did not pass, so the rung 7 profile was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-profile --rung 8 --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r8-full.json; else echo "SKIPPED: the quick check did not pass, so the rung 8 profile was not run"; fi

In [ ]:
# 6. The digest: A is R7, B is R8. Paste this cell's output back.
!if [ -f results/gpu-r7-full.json ] && [ -f results/gpu-r8-full.json ]; then tissuemc-profile --digest results/gpu-r7-full.json results/gpu-r8-full.json; else echo "SKIPPED: a profile above did not finish"; fi

In [ ]:
import sys

sys.path.insert(0, "python")
import json
import pathlib
import subprocess


def load(name):
    p = pathlib.Path("results") / name
    return json.loads(p.read_text()) if p.exists() else None


def bench_summary(b):
    if not b:
        return None
    return {
        "median_kernel_s": round(b["kernel"]["median"], 3),
        "kernel_seconds": [round(x, 3) for x in b["kernel_seconds"]],
        "million_events_per_s": round(b["events_per_second"] / 1e6, 1),
        "validated_first": b["validated_before_measuring"],
        "profiles": b.get("profiles", True),
    }


def cross_summary(c):
    if not c:
        return None
    return {
        "agree": c["agree"],
        "gaps": {g["name"]: [g["value"], g["limit"]] for g in c["gaps"]},
    }


def validation_summary(val):
    if not val:
        return None
    return {
        "checks": len(val),
        "passed": sum(r["passed"] for r in val),
        "failed": [
            f"{r['case']} {r['quantity']} z={r['z']:+.2f} ({r['source']})"
            for r in val
            if not r["passed"]
        ],
    }


def tail(name, n=12):
    p = pathlib.Path("results") / name
    return p.read_text().splitlines()[-n:] if p.exists() else None


commit = subprocess.run(
    ["git", "log", "-1", "--format=%H"], capture_output=True, text=True
).stdout.strip()
r8 = load("gpu-r8.json")
summary = {
    "commit": commit,
    "device": r8["device"] if r8 else None,
    "validation_r8": validation_summary(load("gpu-r8-validation.json")),
    "cross_engine_r7": cross_summary(load("gpu-r7-r8session-cross-engine.json")),
    "cross_engine_r8": cross_summary(load("gpu-r8-cross-engine.json")),
    "benchmark": {
        "r7": bench_summary(load("gpu-r7-r8session.json")),
        "r8": bench_summary(r8),
        "r6_noprofiles": bench_summary(load("gpu-r6-noprofiles-r8session.json")),
        "r8_again": bench_summary(load("gpu-r8-again.json")),
    },
    "diagnostics": {
        "quick_check_passed": pathlib.Path("results/.quick_ok").exists(),
        "build_tail": tail("build.log"),
        "quick_check_tail": tail("quick.log", 20),
    },
}
print("===== R8 SUMMARY =====")
print(json.dumps(summary, indent=1))